# Case 3 Demo: Inverse Run

This notebook loads the prepared forward checkpoint and runs the inverse step.

Required files in `github_demo_outputs/`:

- `adr_channel_demo.pkl`
- `forward_surrogate_state_dict_demo.pt`


In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import pickle
import time

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
except ImportError as exc:
    raise ImportError(
        "This notebook requires PyTorch. Install torch before running the demo."
    ) from exc


SEED = 7
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_default_dtype(torch.float32)

OUTPUT_DIR = Path("github_demo_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
FORWARD_FIELDS_PKL = OUTPUT_DIR / "adr_channel_demo.pkl"
FORWARD_CKPT = OUTPUT_DIR / "forward_surrogate_state_dict_demo.pt"

if not FORWARD_FIELDS_PKL.exists():
    raise FileNotFoundError(
        f"Missing forward reference field: {FORWARD_FIELDS_PKL}. "
        "Put the prepared demo files in github_demo_outputs/ before running."
    )

if not FORWARD_CKPT.exists():
    raise FileNotFoundError(
        f"Missing forward surrogate checkpoint: {FORWARD_CKPT}. "
        "Put the prepared demo files in github_demo_outputs/ before running."
    )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Loading forward reference from: {FORWARD_FIELDS_PKL.resolve()}")
print(f"Loading forward checkpoint from: {FORWARD_CKPT.resolve()}")

mpl.rcParams.update(
    {
        "font.family": "serif",
        "font.serif": ["Times New Roman", "DejaVu Serif"],
        "mathtext.fontset": "cm",
        "axes.labelsize": 13,
        "axes.titlesize": 13,
        "axes.linewidth": 0.8,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "xtick.direction": "in",
        "ytick.direction": "in",
        "xtick.major.size": 4,
        "ytick.major.size": 4,
        "legend.fontsize": 11,
        "figure.dpi": 150,
        "savefig.dpi": 300,
    }
)


Using device: cpu
Loading forward reference from: /scratch/projects/CFP04/CFP04-CF-050/tmp_case3_exec_ygtN7L/github_demo_outputs/adr_channel_demo.pkl
Loading forward checkpoint from: /scratch/projects/CFP04/CFP04-CF-050/tmp_case3_exec_ygtN7L/github_demo_outputs/forward_surrogate_state_dict_demo.pt


In [2]:
class DemoConfig:
    pass


@dataclass
class InverseConfig:
    inverse_steps: int = 150
    inverse_lr: float = 2e-2
    inverse_patience: int = 120
    inverse_min_delta: float = 1e-4
    inverse_metric_alpha: float = 1.0
    inverse_metric_beta: float = 1.0
    enable_early_stop: bool = False


with FORWARD_FIELDS_PKL.open("rb") as f:
    data = pickle.load(f)

A_ref, B_ref, C_ref = data["A"], data["B"], data["C"]
xv, yv = data["x"], data["y"]
cfg = data["cfg"]
inverse_cfg = InverseConfig()
for key, value in inverse_cfg.__dict__.items():
    setattr(cfg, key, value)

Ny, Nx = A_ref.shape
x = np.broadcast_to(xv[None, :], (Ny, Nx))
y = np.broadcast_to(yv[:, None], (Ny, Nx))

k_scale = 1e-6
k_s_true = cfg.k / k_scale

X_single = np.stack(
    [
        np.full_like(A_ref, k_s_true, dtype=np.float32),
        np.full_like(A_ref, cfg.mA, dtype=np.float32),
        np.full_like(A_ref, cfg.mB, dtype=np.float32),
        (x / cfg.L).astype(np.float32),
        (y / cfg.H).astype(np.float32),
    ],
    axis=0,
)[None, ...]

Y_single = np.stack([A_ref, B_ref, C_ref], axis=0)[None, ...].astype(np.float32)
Y_mean = Y_single.mean(axis=(0, 2, 3), keepdims=True)
Y_std = Y_single.std(axis=(0, 2, 3), keepdims=True) + 1e-8


def to_torch(x):
    return torch.tensor(x, dtype=torch.float32, device=device)


def denorm_torch(y_norm):
    return y_norm * to_torch(Y_std) + to_torch(Y_mean)


A_mean, A_std = float(Y_mean[0, 0, 0, 0]), float(Y_std[0, 0, 0, 0])
Aref_n_t = to_torch(((A_ref - A_mean) / A_std)[None, None, ...])


class SpectralConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, modes_x=24, modes_y=12):
        super().__init__()
        self.modes_x = modes_x
        self.modes_y = modes_y
        scale = 1.0 / (in_channels * out_channels)
        self.weight = nn.Parameter(
            scale * torch.randn(in_channels, out_channels, modes_y, modes_x, dtype=torch.cfloat)
        )

    def forward(self, x):
        batch_size, _, ny, nx = x.shape
        x_ft = torch.fft.rfft2(x, norm="ortho")
        ky = min(self.modes_y, ny)
        kx = min(self.modes_x, nx // 2 + 1)
        out_ft = torch.zeros(
            batch_size,
            self.weight.shape[1],
            ny,
            nx // 2 + 1,
            dtype=torch.cfloat,
            device=x.device,
        )
        out_ft[:, :, :ky, :kx] = torch.einsum(
            "bcpq,copq->bopq", x_ft[:, :, :ky, :kx], self.weight[:, :, :ky, :kx]
        )
        return torch.fft.irfft2(out_ft, s=(ny, nx), norm="ortho").real


class FNOBlock(nn.Module):
    def __init__(self, width, modes_x=24, modes_y=12):
        super().__init__()
        self.spectral = SpectralConv2d(width, width, modes_x, modes_y)
        self.w = nn.Conv2d(width, width, 1)
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(self.spectral(x) + self.w(x))


class FNO2D(nn.Module):
    def __init__(self, in_channels=5, out_channels=3, width=48, depth=6, modes_x=24, modes_y=12):
        super().__init__()
        self.fc0 = nn.Conv2d(in_channels, width, 1)
        self.blocks = nn.ModuleList([FNOBlock(width, modes_x, modes_y) for _ in range(depth)])
        self.fc1 = nn.Conv2d(width, width, 1)
        self.fc2 = nn.Conv2d(width, out_channels, 1)
        self.act = nn.GELU()

    def forward(self, x):
        x = self.fc0(x)
        for block in self.blocks:
            x = block(x)
        x = self.act(self.fc1(x))
        return self.fc2(x)


dx = cfg.L / (cfg.Nx - 1)
dy = cfg.H / (cfg.Ny - 1)
u = to_torch(cfg.u)
D = to_torch(cfg.D)
Awall = to_torch(cfg.A_wall)
Bwall = to_torch(cfg.B_wall)
mA_t = to_torch(cfg.mA)
mB_t = to_torch(cfg.mB)


def ddx_upwind2(phi):
    der = torch.zeros_like(phi)
    der[:, :, 2:] = (3 * phi[:, :, 2:] - 4 * phi[:, :, 1:-1] + phi[:, :, :-2]) / (2 * dx)
    der[:, :, 1] = (phi[:, :, 1] - phi[:, :, 0]) / dx
    der[:, :, 0] = der[:, :, 1]
    return der


def laplace_center2(phi):
    out = torch.zeros_like(phi)
    out[:, 1:-1, 1:-1] = (
        (phi[:, 1:-1, 2:] - 2 * phi[:, 1:-1, 1:-1] + phi[:, 1:-1, :-2]) / (dx * dx)
        + (phi[:, 2:, 1:-1] - 2 * phi[:, 1:-1, 1:-1] + phi[:, :-2, 1:-1]) / (dy * dy)
    )
    return out


def pde_residual_loss(Y_phys, k_s, mA, mB):
    k = k_s * k_scale
    A = Y_phys[:, 0]
    B = Y_phys[:, 1]
    C = Y_phys[:, 2]
    reaction = k * (A.clamp_min(0.0) ** mA) * (B.clamp_min(0.0) ** mB)
    RA = -u * ddx_upwind2(A) + D * laplace_center2(A) - reaction
    RB = -u * ddx_upwind2(B) + D * laplace_center2(B) - reaction
    RC = -u * ddx_upwind2(C) + D * laplace_center2(C) + reaction
    return (
        (RA[:, 1:-1, 1:-1] ** 2).mean()
        + (RB[:, 1:-1, 1:-1] ** 2).mean()
        + (RC[:, 1:-1, 1:-1] ** 2).mean()
    )


def bc_loss(Y_phys):
    A = Y_phys[:, 0]
    B = Y_phys[:, 1]
    C = Y_phys[:, 2]
    loss_in = (A[:, :, 0] ** 2).mean() + (B[:, :, 0] ** 2).mean() + (C[:, :, 0] ** 2).mean()
    loss_out = (
        ((A[:, :, -1] - A[:, :, -2]) ** 2).mean()
        + ((B[:, :, -1] - B[:, :, -2]) ** 2).mean()
        + ((C[:, :, -1] - C[:, :, -2]) ** 2).mean()
    )
    loss_bottom = (
        ((A[:, 0, :] - Awall) ** 2).mean()
        + ((B[:, 1, :] - B[:, 0, :]) ** 2).mean()
        + ((C[:, 1, :] - C[:, 0, :]) ** 2).mean()
    )
    loss_top = (
        ((B[:, -1, :] - Bwall) ** 2).mean()
        + ((A[:, -1, :] - A[:, -2, :]) ** 2).mean()
        + ((C[:, -1, :] - C[:, -2, :]) ** 2).mean()
    )
    return loss_in + loss_out + loss_bottom + loss_top


model = FNO2D().to(device)
state_dict = torch.load(FORWARD_CKPT, map_location=device)
model.load_state_dict(state_dict)
model.eval()

for p in model.parameters():
    p.requires_grad = False

print("Forward surrogate checkpoint loaded and frozen.")


/tmp/ipykernel_870481/4253794231.py:185: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(FORWARD_CKPT, map_location=device)


Forward surrogate checkpoint loaded and frozen.


In [3]:
logs_k = nn.Parameter(torch.tensor(np.log10(0.1e-6 / k_scale), device=device))
inverse_optimizer = optim.Adam([logs_k], lr=cfg.inverse_lr)

lambda_data = 25.0
lambda_pde = 0.5e-2
lambda_bc = 1e-3

history = []
c_out_ref = float(C_ref[:, -1].mean())
best_metric = float("inf")
best_rel_c_out_error = float("inf")
best_field_loss = float("inf")
best_step = 0


def build_input(logs_value):
    logs_clamped = torch.clamp(logs_value, -2.0, 2.0)
    k_s = torch.pow(10.0, logs_clamped)
    K = k_s.expand(Ny, Nx)
    MA = mA_t.expand(Ny, Nx)
    MB = mB_t.expand(Ny, Nx)
    x_ch = to_torch((x / cfg.L).astype(np.float32))
    y_ch = to_torch((y / cfg.H).astype(np.float32))
    Xin = torch.stack([K, MA, MB, x_ch, y_ch], dim=0).unsqueeze(0)
    return Xin, k_s


print(f"Running backward parameter identification for {cfg.inverse_steps} steps...")
start_inverse = time.time()
for step in range(1, cfg.inverse_steps + 1):
    inverse_optimizer.zero_grad()
    Xin, k_s_var = build_input(logs_k)
    pred_norm = model(Xin.float())
    Y_phys = denorm_torch(pred_norm)
    loss_field = torch.mean((pred_norm[:, 0:1] - Aref_n_t) ** 2)
    loss_pde = pde_residual_loss(Y_phys, k_s_var, mA_t, mB_t)
    loss_bc = bc_loss(Y_phys)
    loss_total = lambda_data * loss_field + lambda_pde * loss_pde + lambda_bc * loss_bc
    loss_total.backward()
    inverse_optimizer.step()

    with torch.no_grad():
        k_s_val = float(torch.pow(10.0, logs_k.clamp(-2.0, 2.0)).cpu())
        k_val = k_s_val * k_scale
        c_out_pred = float(Y_phys[:, 2, :, -1].mean().detach().cpu())
        rel_c_out_error = abs(c_out_pred - c_out_ref) / (abs(c_out_ref) + 1e-8)
        field_loss_value = float(loss_field.detach().cpu())
        combined_metric = (
            cfg.inverse_metric_alpha * field_loss_value
            + cfg.inverse_metric_beta * rel_c_out_error
        )

        if combined_metric < best_metric - cfg.inverse_min_delta:
            best_metric = combined_metric
            best_rel_c_out_error = rel_c_out_error
            best_field_loss = field_loss_value
            best_step = step

    history.append(
        {
            "loss": float(loss_total.cpu()),
            "field_loss": field_loss_value,
            "k": k_val,
            "c_out": c_out_pred,
            "rel_c_out_error": rel_c_out_error,
            "combined_metric": combined_metric,
        }
    )

    if step % 25 == 0 or step == 1:
        print(
            f"[Inverse {step:04d}/{cfg.inverse_steps}] "
            f"field={loss_field:.3e} pde={loss_pde:.3e} bc={loss_bc:.3e} "
            f"total={loss_total:.3e} k={k_val:.3e} "
            f"rel_c_out={rel_c_out_error:.3e} metric={combined_metric:.3e}"
        )

print(f"Inverse stage finished in {time.time() - start_inverse:.2f} s")

with torch.no_grad():
    Xin_final, _ = build_input(logs_k.detach())
    Y_pred_norm = model(Xin_final.float())
    Y_pred = denorm_torch(Y_pred_norm).detach().cpu().numpy()
    k_final = float(torch.pow(10.0, logs_k.clamp(-2.0, 2.0)).cpu()) * k_scale
    c_out_final = float(Y_pred[0, 2, :, -1].mean())
    rel_c_out_final = abs(c_out_final - c_out_ref) / (abs(c_out_ref) + 1e-8)

result = {
    "hist": history,
    "k_id": k_final,
    "c_out_ref": c_out_ref,
    "c_out_id": c_out_final,
    "rel_c_out_final": rel_c_out_final,
    "best_metric": best_metric,
    "best_rel_c_out_error": best_rel_c_out_error,
    "best_field_loss": best_field_loss,
    "best_step": best_step,
    "mA_fixed": float(cfg.mA),
    "mB_fixed": float(cfg.mB),
    "Y_single": Y_single,
    "Y_pred": Y_pred,
    "forward_checkpoint": str(FORWARD_CKPT),
    "note": "Inverse-only demo run from the saved forward checkpoint",
}

result_pkl = OUTPUT_DIR / "adr_pino_result_inverse_only_demo.pkl"
with result_pkl.open("wb") as f:
    pickle.dump(result, f)

print(f"Saved inverse-only result to: {result_pkl}")
print(f"Recovered k = {k_final:.3e}")
print(
    f"Best metric observed = {best_metric:.3e} at step {best_step} "
    f"(field={best_field_loss:.3e}, rel_c_out={best_rel_c_out_error:.3e})"
)


Running backward parameter identification for 150 steps...


[Inverse 0001/150] field=6.009e-03 pde=3.549e-07 bc=5.304e+00 total=1.555e-01 k=1.047e-07 rel_c_out=5.570e-02 metric=6.171e-02


[Inverse 0025/150] field=3.568e-03 pde=3.190e-07 bc=5.156e+00 total=9.437e-02 k=3.358e-07 rel_c_out=4.349e-02 metric=4.705e-02


[Inverse 0050/150] field=3.075e-05 pde=2.564e-07 bc=5.038e+00 total=5.807e-03 k=1.210e-06 rel_c_out=6.086e-03 metric=6.117e-03


[Inverse 0075/150] field=2.900e-05 pde=2.565e-07 bc=5.039e+00 total=5.764e-03 k=1.165e-06 rel_c_out=5.785e-03 metric=5.814e-03


[Inverse 0100/150] field=2.784e-05 pde=2.569e-07 bc=5.041e+00 total=5.737e-03 k=1.218e-06 rel_c_out=4.427e-03 metric=4.455e-03


[Inverse 0125/150] field=2.716e-05 pde=2.566e-07 bc=5.040e+00 total=5.719e-03 k=1.193e-06 rel_c_out=5.214e-03 metric=5.242e-03


[Inverse 0150/150] field=2.697e-05 pde=2.567e-07 bc=5.041e+00 total=5.715e-03 k=1.201e-06 rel_c_out=4.928e-03 metric=4.955e-03
Inverse stage finished in 26.09 s
Saved inverse-only result to: github_demo_outputs/adr_pino_result_inverse_only_demo.pkl
Recovered k = 1.201e-06
Best metric observed = 2.796e-04 at step 64 (field=1.129e-04, rel_c_out=1.667e-04)


In [4]:
with (OUTPUT_DIR / "adr_pino_result_inverse_only_demo.pkl").open("rb") as f:
    result = pickle.load(f)

hist = result["hist"]
k_id = float(result["k_id"])
c_out_ref = float(result["c_out_ref"])
c_out_id = float(result["c_out_id"])
best_metric = float(result["best_metric"])
best_rel_c_out_error = float(result["best_rel_c_out_error"])
best_field_loss = float(result["best_field_loss"])
best_step = int(result["best_step"])
Y_single = np.array(result["Y_single"])
Y_pred = np.array(result["Y_pred"])

A_ref, B_ref, C_ref = Y_single[0]
if Y_pred.ndim == 4:
    Y_pred = Y_pred[0]
A_id, B_id, C_id = Y_pred

fig, axs = plt.subplots(3, 3, figsize=(12, 10))
titles = [
    [r"$A_{ref}$", r"$A_{id}$", r"$|A_{ref} - A_{id}|$"],
    [r"$B_{ref}$", r"$B_{id}$", r"$|B_{ref} - B_{id}|$"],
    [r"$C_{ref}$", r"$C_{id}$", r"$|C_{ref} - C_{id}|$"],
]
fields = [
    [A_ref, A_id, np.abs(A_ref - A_id)],
    [B_ref, B_id, np.abs(B_ref - B_id)],
    [C_ref, C_id, np.abs(C_ref - C_id)],
]

pair_limits = [
    (float(min(A_ref.min(), A_id.min())), float(max(A_ref.max(), A_id.max()))),
    (float(min(B_ref.min(), B_id.min())), float(max(B_ref.max(), B_id.max()))),
    (float(min(C_ref.min(), C_id.min())), float(max(C_ref.max(), C_id.max()))),
]
err_limits = [
    (0.0, float(np.abs(A_ref - A_id).max())),
    (0.0, float(np.abs(B_ref - B_id).max())),
    (0.0, float(np.abs(C_ref - C_id).max())),
]

for i in range(3):
    for j in range(3):
        if j < 2:
            vmin, vmax = pair_limits[i]
            cmap = "turbo"
        else:
            vmin, vmax = err_limits[i]
            cmap = "bwr"
        im = axs[i, j].imshow(
            fields[i][j],
            origin="lower",
            cmap=cmap,
            aspect="auto",
            vmin=vmin,
            vmax=vmax,
        )
        axs[i, j].set_box_aspect(0.25)
        axs[i, j].set_title(titles[i][j], pad=4)
        axs[i, j].axis("off")
        axs[i, j].text(
            0.02,
            0.92,
            f"({chr(97 + 3 * i + j)})",
            transform=axs[i, j].transAxes,
            fontsize=12,
            fontweight="bold",
            color="white",
            ha="left",
            va="top",
        )
        cbar = plt.colorbar(im, ax=axs[i, j], fraction=0.035, pad=0.01)
        cbar.formatter.set_powerlimits((-2, 2))
        cbar.ax.yaxis.get_offset_text().set(size=11)
        cbar.update_ticks()

plt.tight_layout(pad=0.5)
field_fig = OUTPUT_DIR / "field_comparison_inverse_only_demo.svg"
plt.savefig(field_fig, format="svg", bbox_inches="tight")
plt.show()

k_series = np.array([row["k"] for row in hist], dtype=float)
field_loss_series = np.array([row["field_loss"] for row in hist], dtype=float)
delta_c_series = np.array([row["rel_c_out_error"] for row in hist], dtype=float)
iterations = np.arange(1, len(k_series) + 1)

fig, axs = plt.subplots(1, 3, figsize=(12, 4.5))

axs[0].plot(iterations, k_series, "k-x", lw=1.2, label=r"$k^{(n)}$")
axs[0].set_xlabel("Backward iterations")
axs[0].set_ylabel(r"$k$")
axs[0].text(0.89, 0.93, "(a)", transform=axs[0].transAxes, fontsize=14, fontweight="bold")

axs[1].semilogy(iterations, field_loss_series, "k-x", lw=1.2, label="field loss")
axs[1].set_xlabel("Backward iterations")
axs[1].set_ylabel("Field loss")
axs[1].legend(frameon=False, loc="lower right")
axs[1].text(0.89, 0.93, "(b)", transform=axs[1].transAxes, fontsize=14, fontweight="bold")

axs[2].semilogy(
    iterations,
    delta_c_series,
    color="k",
    marker="^",
    mfc="none",
    mec="b",
    ms=6,
    lw=1.2,
    label=r"$|\Delta \overline{C}_{C,out}^{(n)}|$",
)
axs[2].set_xlabel("Backward iterations")
axs[2].set_ylabel(r"$|\Delta \overline{C}_{C,out}|$")
axs[2].legend(frameon=False, loc="lower right")
axs[2].text(0.89, 0.93, "(c)", transform=axs[2].transAxes, fontsize=14, fontweight="bold")

plt.tight_layout(pad=1.5, w_pad=2)
conv_fig = OUTPUT_DIR / "triple_panel_identification_inverse_only_demo.svg"
plt.savefig(conv_fig, format="svg", bbox_inches="tight")
plt.show()

print(f"Field comparison figure: {field_fig}")
print(f"Convergence figure: {conv_fig}")
print(f"Final identified k: {k_id:.3e}")
print(f"Reference outlet average C: {c_out_ref:.3e}")
print(f"Identified outlet average C: {c_out_id:.3e}")
print(
    f"Best metric observed: {best_metric:.3e} at step {best_step} "
    f"(field={best_field_loss:.3e}, rel_c_out={best_rel_c_out_error:.3e})"
)


Field comparison figure: github_demo_outputs/field_comparison_inverse_only_demo.svg
Convergence figure: github_demo_outputs/triple_panel_identification_inverse_only_demo.svg
Final identified k: 1.201e-06
Reference outlet average C: 5.766e-02
Identified outlet average C: 5.738e-02
Best metric observed: 2.796e-04 at step 64 (field=1.129e-04, rel_c_out=1.667e-04)
